In [1]:
import sys
from pathlib import Path

# Support running notebook from either project root or notebooks/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for base in (cwd, cwd.parent):
    if (base / "models").exists():
        PROJECT_ROOT = base
        if str(base) not in sys.path:
            sys.path.insert(0, str(base))
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root (folder containing 'models').")

In [2]:
import time
from typing import Tuple

import torch
from IPython.display import Latex, display

from evaluation import calculate_average_log_probability
from generate import Generator_Qwen_3
from models import Qwen_3_Model, get_qwen3_config
from utils import get_device, print_generate_stats

In [3]:
config = get_qwen3_config("0.6B")

model = Qwen_3_Model(**config)

device = get_device()
model.to(device)

checkpoint_path = PROJECT_ROOT / "checkpoints" / "qwen3_0.6b_base.pth"
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

state_dict = torch.load(checkpoint_path, map_location=device)
if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
    state_dict = state_dict["model_state_dict"]

model.load_state_dict(state_dict)

model = torch.compile(model)

Using MPS device


In [4]:
generator = Generator_Qwen_3(
    model=model,
    num_layers=config["num_layers"],
    tokenizer_file_path=PROJECT_ROOT / "tokenizer" / "qwen_3_instruct_tokenizer.json",
    model_type="base",
    apply_chat_template=False,
    add_generation_prompt=False,
    add_thinking=False,
)

In [7]:
def generate_text_response(
    prompt: str,
    max_token_length: int = 200,
    temperature: float = 0.7,
    top_k: int = 50,
    top_p: float = 0.9,
    stats: bool = False,
    tok_probs: bool = False,
) -> Tuple[str, float]:
    input_ids = generator.text_to_token_ids(
        text=prompt,
    ).unsqueeze(0)

    start_time = time.time()
    output_ids = generator.generate(
        idx=input_ids,
        max_token_length=max_token_length,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        cache_enabled=True,
    )
    end_time = time.time()

    if stats:
        print_generate_stats(output_ids[len(input_ids) :], start_time, end_time)
    average_log_probability = None
    if tok_probs:
        average_log_probability = calculate_average_log_probability(
            model=model,
            prompt_idx=input_ids,
            idx=output_ids,
            device=device,
            show=False,
        )

    response = generator.token_ids_to_text(output_ids.squeeze(0))

    return response, average_log_probability

In [9]:
prompt = (
    r"If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?"
)

response, average_log_probability = generate_text_response(
    prompt=prompt,
    max_token_length=500,
    # temperature=0.7,
    # top_k=50,
    # top_p=0.9,
    temperature=None,
    top_k=None,
    top_p=None,
    stats=False,
    tok_probs=False,
)

print(response, average_log_probability)

If $a+b=3$ and $ab=\tfrac{13}{6}$, what is the value of $a^2+b^2$? To find the value of \( a^2 + b^2 \) given that \( a + b = 3 \) and \( ab = \frac{13}{6} \), we can use the following algebraic identity:

\[
a^2 + b^2 = (a + b)^2 - 2ab
\]

**Step 1:** Substitute the given values into the equation.

\[
a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
\]

**Step 2:** Calculate \( (3)^2 \).

\[
(3)^2 = 9
\]

**Step 3:** Calculate \( 2 \times \frac{13}{6} \).

\[
2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3}
\]

**Step 4:** Subtract the second result from the first.

\[
a^2 + b^2 = 9 - \frac{13}{3} = 9 - \frac{1}{3} = \frac{27}{3}
\]

Therefore, the value of \( a^2 + b^2 \) is \(\boxed{\frac{2}{3}\}. None


In [10]:
display(Latex(response))

<IPython.core.display.Latex object>